In [2]:
import pandas as pd
import numpy as np 
import tensorflow as tf 
import pickle
import re 
from sklearn.preprocessing import LabelEncoder

In [3]:
df=pd.read_csv('spam.csv', encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [4]:
df=df.drop(['Unnamed: 2',	'Unnamed: 3', 	'Unnamed: 4'], axis=1)

In [5]:
df

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [6]:
x=df.drop('v1', axis=1)
y=df['v1']

In [7]:
label=LabelEncoder()
y=label.fit_transform(y)

In [8]:
y

array([0, 0, 1, ..., 0, 0, 0], shape=(5572,))

In [9]:
from  bs4 import BeautifulSoup
def clean_text(text):
    text=text.lower()
    ext = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [10]:
x['v2']=x['v2'].apply(clean_text)

In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer()
tokenizer.fit_on_texts(x['v2'])

In [12]:
sequence=tokenizer.texts_to_sequences(x['v2'])

In [13]:
print(x['v2'].apply(type).value_counts())

v2
<class 'str'>    5572
Name: count, dtype: int64


In [14]:
max_length=500
x=pad_sequences(sequence, maxlen=500, padding='pre')

In [15]:
x

array([[   0,    0,    0, ...,   58, 4310,  137],
       [   0,    0,    0, ...,  444,    6, 1820],
       [   0,    0,    0, ..., 2864,  382, 2865],
       ...,
       [   0,    0,    0, ..., 9411,  231, 9412],
       [   0,    0,    0, ...,  198,   12,   50],
       [   0,    0,    0, ...,    1,   41,  258]],
      shape=(5572, 500), dtype=int32)

In [16]:
empty_idx = [i for i, seq in enumerate(sequence) if len(seq) == 0]
print(empty_idx[:10])
print(df.iloc[empty_idx[:5]])

[3374, 4822]
       v1       v2
3374  ham      :) 
4822  ham  :-) :-)


In [17]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x, y, test_size=0.2, random_state=42)

In [18]:
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding,Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

In [19]:
max_feature=10000

In [20]:
model=Sequential()
model.add(Embedding(max_feature, 128, input_length=max_length))
model.add(Bidirectional(SimpleRNN(128, activation='relu', return_sequences=True)))
model.add(Bidirectional(SimpleRNN(64, activation='relu')))
model.add(Dense(1, activation='sigmoid'))

d:\PROJECTS\DEEP LEARNING PROJECT\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001,
    clipnorm=1.0
)

In [25]:
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

In [26]:
early=EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [27]:
history=model.fit(
    x_train, y_train,
    epochs=25, 
    validation_data=(x_test, y_test),
    callbacks=[early]
)

Epoch 1/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 95s 638ms/step - accuracy: 0.9917 - loss: 0.1288 - val_accuracy: 0.9839 - val_loss: 0.0610
Epoch 2/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 93s 664ms/step - accuracy: 0.8975 - loss: nan - val_accuracy: 0.8655 - val_loss: nan
Epoch 3/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 90s 642ms/step - accuracy: 0.8661 - loss: nan - val_accuracy: 0.8655 - val_loss: nan
Epoch 4/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 515ms/step - accuracy: 0.8661 - loss: nan - val_accuracy: 0.8655 - val_loss: nan
